In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from joblib import dump
from tqdm import tqdm

In [ ]:
df_orig = pd.read_csv(
    r"C:\Users\ebeva\SkyTruth\cv3\source potential\initial_testDS.csv"
).drop_duplicates()

In [ ]:
# sample 10% of the slick data
# df_sample = df_orig.sample(frac=0.1, random_state=42)
# df_remain = df_orig.drop(df_sample.index)

In [ ]:
# df_sample.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_baseline_test.csv", index=False)
# df_remain.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_baseline_train.csv", index=False)

In [ ]:
df_test = pd.read_csv(
    r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_baseline_test.csv"
)
df_train = pd.read_csv(
    r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_baseline_train.csv"
)

In [ ]:
# df_feats_added = pd.read_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\hitl_featsAdded_011426.csv").drop_duplicates()
# df_feats_added_sample = df_feats_added[df_feats_added['id'].isin(df_test['id'])]
# df_feats_added_remain = df_feats_added[df_feats_added['id'].isin(df_train['id'])]
# df_feats_added_sample.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_featsAdded_test.csv", index=False)
# df_feats_added_remain.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_featsAdded_train.csv", index=False)

# df_full = pd.read_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\cerulean_detections_all.csv")
# unnattributed_ids = set(df_full['id']) - set(df_orig['id'])
# test_unnattributed_ids_sample = list(unnattributed_ids)[0:int(len(unnattributed_ids)*0.1)]
# train_unnattributed_ids = list(unnattributed_ids)[int(len(unnattributed_ids)*0.1):]
# df_full_sample = df_full[df_full['id'].isin(df_test['id']) | df_full['id'].isin(test_unnattributed_ids_sample)]
# df_full_remain = df_full[df_full['id'].isin(df_train['id']) | df_full['id'].isin(train_unnattributed_ids)]
# df_full_sample.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_full_test.csv", index=False)
# df_full_remain.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_full_train.csv", index=False)

In [ ]:
# df_tst = pd.read_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_featsAdded_test.csv")
# df = pd.read_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_featsAdded_train.csv")

In [ ]:
df_full_train = pd.read_csv(
    r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_full_train.csv"
)

In [ ]:
# sum(df['is_slick'])

In [ ]:
# feature_columns = ['length', 'area', 'perimeter','polsby_popper', 'fill_factor', 'aspect_ratio_factor']
# X = pd.get_dummies(X, columns=feature_columns)

In [ ]:
# remove half of the is_slick == false from df


In [ ]:
num_of_trials = 1
best_val_accuracy = 0
# df = pd.concat([
#     df_full_train[df_full_train['is_slick'] == 1],
#     df_full_train[df_full_train['is_slick'] == 0].sample(frac=0.5, random_state=42)
# ])
df = df_full_train
feature_columns = [
    "length",
    "area",
    "perimeter",
    "polsby_popper",
    "fill_factor",
    "aspect_ratio_factor",
    "geometry_count",
    "largest_area",
    "median_area",
]
X = df[feature_columns]
y = df["is_slick"].astype(int)
for trial in tqdm(range(num_of_trials)):
    print(f"Trial {trial + 1}/{num_of_trials}")
    X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y)
    # rf = RandomForestClassifier(
    #     n_estimators=500,
    #     min_samples_leaf=5,
    #     class_weight="balanced",
    #     n_jobs=-1,
    #     random_state=42,
    # )
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=15,
        min_samples_split=30,
        max_features=0.5,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    )

    rf.fit(X_train, y_train)
    thres = 0.5
    y_prob = rf.predict_proba(X_val)[:, 1]
    y_pred = (y_prob >= thres).astype(int)
    y_pred_train = rf.predict_proba(X_train)[:, 1]
    y_pred_train = (y_pred_train >= thres).astype(int)
    df["slick_potential"] = rf.predict_proba(X)[:, 1]
    val_accuracy = round(100 * accuracy_score(y_val, y_pred), 6)
    print("Validation Accuracy:", val_accuracy, "%")
    print(
        "Training Accuracy:", round(100 * accuracy_score(y_train, y_pred_train), 6), "%"
    )
    if val_accuracy > best_val_accuracy:
        print("Saving Best model")
        dump(
            rf,
            r"C:\Users\ebeva\SkyTruth\cv3\source potential\models"
            + r"\\"
            + f"random_forest_{int(val_accuracy)}.joblib",
        )

In [ ]:
# rf = load(r"C:\Users\ebeva\SkyTruth\cv3\source potential\models\random_forest_83.joblib")
# df['slick_potential'] = rf.predict_proba(X)[:, 1]

In [ ]:
cm = confusion_matrix(y_val, y_pred)
cm_train = confusion_matrix(y_train, y_pred_train)

In [ ]:
def plot_cm(cm):
    cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots()
    im = ax.imshow(
        cm_normalized, cmap="Blues", alpha=0.6
    )  # <-- alpha sets transparency

    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.ax.set_ylabel("Normalized count", rotation=-90, va="bottom")

    classes = ["Not Slick", "Slick"]
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    for i in range(cm_normalized.shape[0]):
        for j in range(cm_normalized.shape[1]):
            ax.text(
                j,
                i,
                f"{cm[i, j]}\n({cm_normalized[i, j]:.2f})",  # show count + fraction
                ha="center",
                va="center",
                color="black",
                fontsize=12,
            )

    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")
    ax.set_title("Confusion Matrix")
    plt.show()

In [ ]:
plot_cm(cm)

In [ ]:
plot_cm(cm_train)

In [ ]:
163 + 1637 + 139 + 461

In [ ]:
y_prob = rf.predict_proba(X_val)[:, 1]
y_pred = (y_prob >= thres).astype(int)


def metrics_vs_threshold(y_prob, y_true, thresholds=None, plot=True):
    """
    Compute accuracy, precision, recall, and F1-score across a range of thresholds.

    Parameters:
    -----------
    y_prob : array-like
        Predicted probabilities for the positive class.
    y_true : array-like
        True binary labels (0 or 1).
    thresholds : array-like, optional
        Thresholds to evaluate. Defaults to np.linspace(0,1,101)
    plot : bool, optional
        If True, plots the metrics vs threshold.

    Returns:
    --------
    thresholds : np.ndarray
        Threshold values evaluated.
    metrics_dict : dict
        Dictionary with keys 'accuracy', 'precision', 'recall', 'f1', each containing an array of metric values.
    """
    if thresholds is None:
        thresholds = np.linspace(0, 1, 50)

    accuracies, precisions, recalls, f1s = [], [], [], []

    for t in thresholds:
        preds = (y_prob >= t).astype(int)
        accuracies.append(accuracy_score(y_true, preds))
        precisions.append(precision_score(y_true, preds, zero_division=0))
        recalls.append(recall_score(y_true, preds, zero_division=0))
        f1s.append(f1_score(y_true, preds, zero_division=0))

    metrics_dict = {
        "accuracy": np.array(accuracies),
        "precision": np.array(precisions),
        "recall": np.array(recalls),
        "f1": np.array(f1s),
    }

    if plot:
        plt.figure(figsize=(8, 5))
        plt.plot(thresholds, metrics_dict["accuracy"], marker="o", label="Accuracy")
        plt.plot(thresholds, metrics_dict["precision"], marker="x", label="Precision")
        plt.plot(thresholds, metrics_dict["recall"], marker="s", label="Recall")
        plt.plot(thresholds, metrics_dict["f1"], marker="^", label="F1-score")
        plt.xlabel("Threshold")
        plt.ylabel("Metric value")
        plt.title("Metrics vs Threshold")
        plt.grid(True)
        plt.legend()
        # optional: show current threshold if defined
        try:
            plt.axvline(
                thres, color="gray", linestyle="--", label=f"current thres={thres}"
            )
            plt.legend()
        except NameError:
            pass
        plt.show()

    return thresholds, metrics_dict


# compute and plot using the thresholds and y_prob already defined
thresholds, metrics_dict = metrics_vs_threshold(y_prob, y_val)
metric = "precision"
best_idx = np.argmax(metrics_dict["precision"])
print(
    f"Best threshold: {thresholds[best_idx]:.3f} -> {metric}: {metrics_dict[metric][best_idx]:.4f}"
)

In [ ]:
y_prob = rf.predict_proba(X_val)[:, 1]
y_pred = (y_prob >= thresholds[best_idx]).astype(int)

cm = confusion_matrix(y_val, y_pred)
plot_cm(cm)

In [ ]:
# df['prediction']

In [ ]:
import geopandas as gpd
from shapely import wkt

geo_file = r"C:\Users\ebeva\SkyTruth\cv3\source potential\allHITL-1767888944318.csv"
df2 = pd.read_csv(geo_file)
df2["st_astext"] = df2["st_astext"].apply(wkt.loads)
hitl_gdf = gpd.GeoDataFrame(
    df2,
    geometry="st_astext",
    crs="EPSG:4326",  # adjust if coordinates are not lon/lat
)
hitl_gdf["perim_length_ratio"] = hitl_gdf["perimeter"] / hitl_gdf["length"]
hitl_gdf = hitl_gdf[hitl_gdf["hitl_cls"].isin([1, 6, 7, 8])]
hitl_gdf["is_slick"] = hitl_gdf["hitl_cls"].isin([6, 7, 8])

# # Ensure CRS is geographic (EPSG:4326)
# hitl_gdf = hitl_gdf.to_crs(epsg=4326)

# cmap = ListedColormap(["red", "green"])  # 0 = red, 1 = green

# fig, ax = plt.subplots(figsize=(10, 6))

# hitl_gdf.plot(
#     ax=ax,
#     column="is_slick",
#     cmap=cmap,
#     legend=True,
# )

# # Set Mediterranean bounds
# ax.set_xlim(0, 5)
# ax.set_ylim(36, 42)

# ax.set_title("HITL Annotations – Mediterranean Sea")
# plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    rf,
    X_val,
    y_val,
    n_repeats=20,
    random_state=42,
    scoring="f1",  # or "recall", "precision"
)

perm_importance = pd.DataFrame(
    {
        "feature": X_val.columns,
        "importance": result.importances_mean,
        "std": result.importances_std,  # optional but very useful
    }
)

In [ ]:
# horizontal bar plot of permutation importances with error bars
df_pi = perm_importance.sort_values("importance", ascending=True)

plt.figure(figsize=(8, 4))
plt.barh(
    df_pi["feature"], df_pi["importance"], xerr=df_pi["std"], color="C0", alpha=0.8
)
for i, (imp, std) in enumerate(zip(df_pi["importance"], df_pi["std"])):
    plt.text(imp + std + 0.002, i, f"{imp:.3f}", va="center")
plt.xlabel("Mean permutation importance (f1)")
plt.title("Permutation Feature Importances")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# add max_source_collated_score from hitl_gdf to df using 'id' join
score_series = hitl_gdf.drop_duplicates(subset='id').set_index('id')['max_source_collated_score']
df['max_source_collated_score'] = df['id'].map(score_series)
df['max_source_collated_score'] = df['max_source_collated_score'].fillna(-5)
print("Added column; missing values:", df['max_source_collated_score'].isna().sum())
# df[['id', 'max_source_collated_score']].head()

In [ ]:
# exclude sentinel/missing values (you filled missing with -5 earlier)
mask_valid = df['max_source_collated_score'] != -5

# dynamic bins over the valid range
vals = df.loc[mask_valid, 'max_source_collated_score']
bins = np.linspace(vals.min(), vals.max(), 50)

plt.figure(figsize=(8,5))
plt.hist(df.loc[mask_valid & df['is_slick'], 'max_source_collated_score'].dropna(),
         bins=bins, density=True, alpha=0.6, label='Slick (True)', color='green')
plt.hist(df.loc[mask_valid & ~df['is_slick'], 'max_source_collated_score'].dropna(),
         bins=bins, density=True, alpha=0.6, label='Not Slick (False)', color='red')

plt.xlabel('max_source_collated_score')
plt.ylabel('Density')
plt.title('Distribution of max_source_collated_score by is_slick')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
bins = np.linspace(0, 1, 50)

plt.figure(figsize=(8,5))
plt.hist(df.loc[df['is_slick'], 'slick_potential'].dropna(), bins=bins, density=True,
         alpha=0.6, label='Slick (True)', color='green')
plt.hist(df.loc[~df['is_slick'], 'slick_potential'].dropna(), bins=bins, density=True,
         alpha=0.6, label='Not Slick (False)', color='red')

plt.xlabel('slick_potential')
plt.ylabel('Density')
plt.title('Distribution of slick_potential by is_slick')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# scatter: slick_potential vs max_source_collated_score, green for True, red for False
# mask_valid = df['max_source_collated_score'] != -5  # exclude sentinel missing value
df_plot = df

plt.figure(figsize=(8, 6))
plt.scatter(
    df_plot.loc[df_plot["is_slick"], "slick_potential"],
    df_plot.loc[df_plot["is_slick"], "max_source_collated_score"],
    c="green",
    alpha=0.6,
    s=30,
    label="Slick (True)",
)
plt.scatter(
    df_plot.loc[~df_plot["is_slick"], "slick_potential"],
    df_plot.loc[~df_plot["is_slick"], "max_source_collated_score"],
    c="red",
    alpha=0.6,
    s=30,
    label="Not Slick (False)",
)
plt.xlabel("slick_potential")
plt.ylabel("max_source_collated_score")
plt.title("slick_potential vs max_source_collated_score")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# slick_set = df[df['is_slick']].sort_values(by='slick_potential')
# not_slick_set = df[df['is_slick']==False].sort_values(by='slick_potential', ascending=False)
# for i in range(0,30):
#     hitl_gdf[hitl_gdf['id'] == slick_set.iloc[i]['id']].plot()

# for i in range(0,30):
#     sid = not_slick_set.iloc[i]['id']
#     max_col = hitl_gdf[hitl_gdf['id'] == sid]['max_source_collated_score']
#     print(f'https://cerulean.skytruth.org/slicks/{sid}', "Max Coll Score:", max_col.values[0])
#     # hitl_gdf[hitl_gdf['id'] == sid].plot()
# wakes = [3669393, 4486051, 3091272, 3953052, 3789088]

# for i in range(0,30):
#     sid = not_slick_set.iloc[i]['id']
#     max_col = hitl_gdf[hitl_gdf['id'] == sid]['max_source_collated_score']
#     print(f'https://cerulean.skytruth.org/slicks/{sid}', "Max Coll Score:", max_col.values[0])
# hitl_gdf[hitl_gdf['id'] == sid].plot()